# Script 3 — Treinamento dos Modelos de ML (V5)
**TCC: Predição de Indicadores Financeiros Corporativos com ML e IA Generativa**

| Decisão | Justificativa |
|---------|---------------|
| **9 targets** | 3 primários (DRE) + 5 balanço + 1 caixa → habilita Z''-Score prospectivo completo |
| **Split temporal** | Treino ≤2022, Teste 2023–2024 — o modelo nunca viu dados futuros |
| **GroupKFold por empresa** | Evita vazamento temporal cruzado entre empresas no CV |
| **Transformação seletiva por target** | log1p para séries positivas e arcsinh para séries negativas/mistas |
| **SMAPE como métrica principal** | Definido para negativos (Lucro Líquido pode ser negativo) |
| **Theil's U** | Prova que ML supera baseline ingênua |
| **Acurácia Direcional** | Percentual de acertos na direção (sobe/desce) |
| **Imputer no Pipeline** | Evita leakage de NaN das features YoY no CV |
| **Curvas de aprendizado** | Diagnóstico automático de overfitting/underfitting |
| **Análise de resíduos** | Valida pressupostos e detecta padrões sistemáticos |

## Etapa 0 — Dependências e configuração

In [6]:
import logging, json, pickle, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, GridSearchCV, learning_curve
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)

PASTA_SAIDA = Path('outputs')
PASTA_SAIDA.mkdir(exist_ok=True)

logger = logging.getLogger('pipeline_modelagem_v5')
logger.setLevel(logging.DEBUG)
logger.handlers.clear()
_fmt = logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s',
                         datefmt='%Y-%m-%d %H:%M:%S')
_sh = logging.StreamHandler()
_sh.setLevel(logging.INFO)
_sh.setFormatter(_fmt)
logger.addHandler(_sh)
_fh = logging.FileHandler(PASTA_SAIDA / 'pipeline_modelagem_v5.log',
                          mode='w', encoding='utf-8')
_fh.setLevel(logging.DEBUG)
_fh.setFormatter(_fmt)
logger.addHandler(_fh)

# ── Parâmetros ─────────────────────────────────────────────────────────────
N_SPLITS_EXT = 5
N_SPLITS_INT = 5
RANDOM_STATE = 42
ANO_CORTE    = 2022   # treino ≤ 2022, teste ≥ 2023

# Targets que recebem log1p (séries positivas e assimétricas)
LOG_TARGETS = {
    'TARGET_DRE_3.01', 'TARGET_EBITDA',
    'TARGET_BPA_1', 'TARGET_BPA_1.01',
    'TARGET_BPP_2.01', 'TARGET_BPP_2.03', 'TARGET_BPP_2',
}

# Targets com valores negativos ou mistos recebem arcsinh
ARCSINH_TARGETS = {
    'TARGET_DFC_MI_6.01',
    'TARGET_DRE_3.11',  # Lucro Líquido pode ser negativo
}

# Alias retrocompatível para qualquer alvo com transformação
TRANSFORM_TARGETS = LOG_TARGETS | ARCSINH_TARGETS


def get_target_transform(target):
    """
    Retorna a transformação do target:
    - 'log1p'   para séries positivas e assimétricas
    - 'arcsinh' para séries negativas ou mistas
    - 'none'    caso o target não receba transformação
    """
    if target in LOG_TARGETS:
        return 'log1p'
    if target in ARCSINH_TARGETS:
        return 'arcsinh'
    return 'none'


def target_transform(y, transformacao='none'):
    """
    Aplica a transformação escolhida ao target.

    transformacao:
        - 'log1p'   : apenas para valores > -1
        - 'arcsinh' : válida para qualquer valor real
        - 'none'    : sem transformação
    """
    y_arr = np.asarray(y, dtype=float)

    if not np.isfinite(y_arr).all():
        n_bad = np.size(y_arr) - np.isfinite(y_arr).sum()
        raise ValueError(
            f'target_transform: há {n_bad} valores não finitos antes da transformação.'
        )

    if transformacao == 'log1p':
        if np.any(y_arr <= -1):
            n_invalidos = int(np.sum(y_arr <= -1))
            raise ValueError(
                f"target_transform(log1p): {n_invalidos} valores <= -1 encontrados. "
                f"Use 'arcsinh' para targets com negativos/mistos."
            )
        return np.log1p(y_arr)

    if transformacao == 'arcsinh':
        return np.arcsinh(y_arr)

    return y_arr.copy()


def target_inverse_transform(y_pred, transformacao='none'):
    """Inverte a transformação aplicada ao target."""
    y_arr = np.asarray(y_pred, dtype=float)

    if transformacao == 'log1p':
        return np.expm1(y_arr)
    if transformacao == 'arcsinh':
        return np.sinh(y_arr)
    return y_arr


logger.info("Script 3 V5 iniciado | sklearn=%s", __import__('sklearn').__version__)
print("✅ Dependências carregadas")

2026-05-05 10:52:10 | INFO     | Script 3 V5 iniciado | sklearn=1.8.0


✅ Dependências carregadas


## Etapa 1 — Carregamento e split temporal

In [7]:
dataset = pd.read_parquet(PASTA_SAIDA / 'dataset_preparado.parquet')

with open(PASTA_SAIDA / 'features.pkl',      'rb') as f: FEATURES      = pickle.load(f)
with open(PASTA_SAIDA / 'targets.pkl',       'rb') as f: TARGETS       = pickle.load(f)
with open(PASTA_SAIDA / 'kpis.pkl',          'rb') as f: KPIS          = pickle.load(f)
with open(PASTA_SAIDA / 'grupos_treino.pkl', 'rb') as f: GRUPOS_TREINO = pickle.load(f)

# ── Carregar splits do Script 2 (já com 3 conjuntos: treino/teste/prospectivo) ──
# O Script 2 grava treino.parquet e teste.parquet com o split temporal correto.
# O Script 3 consome esses arquivos diretamente — sem recriar o split —
# para garantir que treino ≤ 2022, teste 2023–2024, prospectivo ≥ 2025.
cam_treino = PASTA_SAIDA / 'treino.parquet'
cam_teste  = PASTA_SAIDA / 'teste.parquet'

if cam_treino.exists() and cam_teste.exists():
    treino = pd.read_parquet(cam_treino)
    teste  = pd.read_parquet(cam_teste)
    logger.info("Splits carregados dos parquets do Script 2")
else:
    # Fallback: Script 2 não foi rodado — recalcular com 3 conjuntos
    logger.warning("treino.parquet não encontrado — recalculando split temporal")
    if 'split' not in dataset.columns:
        dataset['split'] = np.where(
            dataset['ANO'].astype(float) <= ANO_CORTE, 'treino',
            np.where(
                dataset['ANO'].astype(float) <= 2024, 'teste',
                'prospectivo'
            )
        )
    treino = dataset[dataset['split'] == 'treino'].reset_index(drop=True)
    teste  = dataset[dataset['split'] == 'teste'].reset_index(drop=True)
    GRUPOS_TREINO = treino['CNPJ_CIA'].values
    treino.to_parquet(cam_treino, index=False)
    teste.to_parquet(cam_teste,   index=False)
    with open(PASTA_SAIDA / 'grupos_treino.pkl', 'wb') as f:
        pickle.dump(GRUPOS_TREINO, f)

# Validação
assert len(treino) > 0, "Treino vazio — verifique o Script 2"
assert len(teste)  > 0, "Teste vazio — verifique o Script 2"

# Verificar que o conjunto prospectivo NÃO vazou para treino ou teste
anos_treino = set(treino['ANO'].dropna().astype(int).unique())
anos_teste  = set(teste['ANO'].dropna().astype(int).unique())
anos_prosp  = {a for a in anos_treino | anos_teste if a >= 2025}
if anos_prosp:
    logger.error("Anos prospectivos vazaram para treino/teste: %s", anos_prosp)
else:
    logger.info("Isolamento prospectivo: PASSOU ✅ (nenhum ano ≥2025 no treino/teste)")

n_tot = len(treino) + len(teste)
logger.info("Treino: %d obs (≤%d) | Teste: %d obs (%d–%d)",
            len(treino), ANO_CORTE,
            len(teste),  ANO_CORTE+1, 2024)

print(f"\n{'='*65}")
print(f"  Split temporal — carregado do Script 2")
print(f"{'='*65}")
print(f"  Treino (≤{ANO_CORTE}): {len(treino):>4} obs | "
      f"DFP={(treino['ORIGEM']=='DFP').sum():>3} | "
      f"ITR={(treino['ORIGEM']=='ITR').sum():>3} | "
      f"anos {sorted(anos_treino)[:3]}...{sorted(anos_treino)[-1:]}")
print(f"  Teste ({ANO_CORTE+1}–2024): {len(teste):>4} obs | "
      f"DFP={(teste['ORIGEM']=='DFP').sum():>3} | "
      f"ITR={(teste['ORIGEM']=='ITR').sum():>3} | "
      f"anos {sorted(anos_teste)}")
print(f"  Prospectivo (≥2025): carregado pelo Script 5")
print(f"  Isolamento prospectivo: ✅")
print(f"{'='*65}")
print(f"\nFeatures: {len(FEATURES)} | Targets: {len(TARGETS)}")
print(f"Targets:")
for t in TARGETS:
    if t in treino.columns:
        # FIX: indicar também targets arcsinh, não apenas log1p
        if t in LOG_TARGETS:
            transform_flag = "📐log   "
        elif t in ARCSINH_TARGETS:
            transform_flag = "📐arcsinh"
        else:
            transform_flag = "         "
        n_tr = treino[t].notna().sum()
        n_te = teste[t].notna().sum() if t in teste.columns else 0
        print(f"  {transform_flag} {t:<30} treino={n_tr:,} | teste={n_te:,}")

2026-05-05 10:52:14 | INFO     | Splits carregados dos parquets do Script 2
2026-05-05 10:52:14 | INFO     | Isolamento prospectivo: PASSOU ✅ (nenhum ano ≥2025 no treino/teste)
2026-05-05 10:52:14 | INFO     | Treino: 713 obs (≤2022) | Teste: 168 obs (2023–2024)



  Split temporal — carregado do Script 2
  Treino (≤2022):  713 obs | DFP=181 | ITR=532 | anos [np.int64(2015), np.int64(2016), np.int64(2017)]...[np.int64(2022)]
  Teste (2023–2024):  168 obs | DFP= 24 | ITR=144 | anos [np.int64(2023), np.int64(2024)]
  Prospectivo (≥2025): carregado pelo Script 5
  Isolamento prospectivo: ✅

Features: 15 | Targets: 9
Targets:
  📐log    TARGET_DRE_3.01                treino=713 | teste=168
  📐arcsinh TARGET_DRE_3.11                treino=713 | teste=168
  📐log    TARGET_EBITDA                  treino=713 | teste=168
  📐log    TARGET_BPA_1                   treino=713 | teste=168
  📐log    TARGET_BPA_1.01                treino=713 | teste=168
  📐log    TARGET_BPP_2.01                treino=713 | teste=168
  📐log    TARGET_BPP_2.03                treino=713 | teste=168
  📐log    TARGET_BPP_2                   treino=713 | teste=168
  📐arcsinh TARGET_DFC_MI_6.01             treino=713 | teste=168


## Etapa 2 — Métricas e Baseline Ingênua

Além das métricas estatísticas clássicas, o pipeline calcula:

**Theil's U:** U<1 prova que o modelo supera a baseline de persistência.
**Acurácia Direcional:** % de acertos na direção (sobe/desce) — mais relevante para gestores.
**SMAPE:** erro percentual simétrico, definido mesmo para valores negativos (Lucro Líquido).

In [8]:
def smape(y_true, y_pred):
    """Symmetric MAPE — definido para valores negativos e próximos de zero."""
    num   = np.abs(y_true - y_pred)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask  = denom > 0
    return float(np.mean(num[mask] / denom[mask])) if mask.sum() > 0 else np.nan


def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def theil_u(y_true, y_pred):
    """
    Coeficiente de desigualdade de Theil (U2).
    U < 1 → modelo supera a baseline ingênua de persistência.
    U = 1 → equivalente à baseline.
    U > 1 → pior que não fazer nada.
    """
    n = len(y_true)
    if n < 2:
        return np.nan
    erro_modelo   = np.sqrt(np.mean((y_true[1:] - y_pred[1:])**2))
    erro_baseline = np.sqrt(np.mean((y_true[1:] - y_true[:-1])**2))
    return float(erro_modelo / erro_baseline) if erro_baseline > 0 else np.nan


def acuracia_direcional(y_true, y_pred):
    """
    Percentual de acertos na direção (sobe/desce) em relação ao período anterior.
    Calculado sobre sequências ordenadas temporalmente.
    """
    if len(y_true) < 2:
        return np.nan
    dir_real = np.sign(np.diff(y_true))
    dir_pred = np.sign(np.diff(y_pred))
    mask = dir_real != 0
    return float(np.mean(dir_real[mask] == dir_pred[mask])) if mask.sum() > 0 else np.nan


def calcular_baseline(treino_df, teste_df, target):
    """
    Baseline ingênua por empresa: persistência do último valor observado
    da própria companhia, respeitando a ordem temporal.

    A previsão de cada linha do teste usa o último valor não-nulo do target
    para a mesma empresa, vindo do histórico do treino ou de períodos anteriores
    do próprio teste. Isso evita mistura entre companhias diferentes e produz
    uma comparação mais justa do que uma baseline global.

    Detecção automática da coluna temporal (prioridade decrescente):
        DT_FIM_EXERC → DATA_REFERENCIA → DATA → DT_REFERENCIA →
        TRIMESTRE → TRI → PERIODO → PERÍODO → ANO

    Se nenhuma coluna temporal estiver disponível, a ordem original é preservada
    dentro de cada empresa (comportamento idêntico a usar ANO como fallback).

    Linhas do teste sem histórico disponível são excluídas da avaliação;
    a métrica de cobertura indica a fração do teste avaliada.
    """
    if target not in treino_df.columns or target not in teste_df.columns:
        return {}

    if 'CNPJ_CIA' not in treino_df.columns or 'CNPJ_CIA' not in teste_df.columns:
        return {}

    # Detecta a coluna temporal mais específica disponível
    candidatos_tempo = [
        'DT_FIM_EXERC', 'DATA_REFERENCIA', 'DATA', 'DT_REFERENCIA',
        'TRIMESTRE', 'TRI', 'PERIODO', 'PERÍODO', 'ANO'
    ]
    time_col = next(
        (c for c in candidatos_tempo
         if c in treino_df.columns and c in teste_df.columns),
        None
    )

    # Colunas de ordenação: empresa + tempo + posição original (desempate estável)
    cols_ordenacao = ['CNPJ_CIA']
    if time_col is not None:
        cols_ordenacao.append(time_col)

    # Preserva a posição original como desempate para garantir estabilidade
    treino_tmp = treino_df.reset_index(drop=True).copy()
    teste_tmp  = teste_df.reset_index(drop=True).copy()
    treino_tmp['_ordem_original'] = np.arange(len(treino_tmp))
    teste_tmp['_ordem_original']  = np.arange(len(teste_tmp))

    cols_select = list(dict.fromkeys(cols_ordenacao + ['_ordem_original', target]))

    base = pd.concat([
        treino_tmp[cols_select].assign(__split='treino'),
        teste_tmp[cols_select].assign(__split='teste'),
    ], ignore_index=True)

    # Ordena dentro de cada empresa para construir a persistência cronológica
    base = base.sort_values(
        cols_ordenacao + ['_ordem_original'], kind='mergesort'
    ).reset_index(drop=True)

    # Último valor observado não-nulo da própria empresa — excluindo o ponto atual
    base['baseline_prev'] = (
        base.groupby('CNPJ_CIA')[target]
            .transform(lambda s: s.ffill().shift(1))
    )

    # Avalia somente o conjunto de teste onde há baseline disponível
    mask_teste   = base['__split'] == 'teste'
    mask_valido  = mask_teste & base[target].notna() & base['baseline_prev'].notna()

    if mask_valido.sum() == 0:
        return {}

    y_t = base.loc[mask_valido, target].values
    y_p = base.loc[mask_valido, 'baseline_prev'].values

    n_teste_total = int(mask_teste.sum())
    cobertura = float(mask_valido.sum() / max(1, n_teste_total))

    return {
        'RMSE_baseline'        : rmse(y_t, y_p),
        'MAE_baseline'         : float(mean_absolute_error(y_t, y_p)),
        'SMAPE_baseline'       : smape(y_t, y_p),
        'R2_baseline'          : float(r2_score(y_t, y_p)),
        'TheilU_baseline'      : 1.0,   # persistência é a referência por definição
        'DA_baseline'          : float(acuracia_direcional(y_t, y_p)),
        'Cobertura_baseline'   : cobertura,
        'TimeCol_baseline'     : time_col if time_col is not None else '',
    }


# Calcular baselines para todos os targets
baselines = {}
print("=== Baseline Ingênua por empresa (persistência) ===")
print(f"  {'Target':<30} {'RMSE':>14} {'SMAPE':>7} {'R²':>6} "
      f"{'TheilU':>7} {'DA':>6} {'Cob.':>6} {'ColTempo'}")
print(f"  {'-'*30} {'-'*14} {'-'*7} {'-'*6} {'-'*7} {'-'*6} {'-'*6} {'-'*14}")
for t in TARGETS:
    b = calcular_baseline(treino, teste, t)
    baselines[t] = b
    if b:
        print(f"  {t:<30} {b['RMSE_baseline']:>14,.0f} "
              f"{b['SMAPE_baseline']:>7.1%} {b['R2_baseline']:>6.3f} "
              f"{b['TheilU_baseline']:>7.2f} {b['DA_baseline']:>6.1%} "
              f"{b.get('Cobertura_baseline', np.nan):>6.1%} "
              f"  {b.get('TimeCol_baseline', 'N/A')}")
    else:
        print(f"  {t:<30} {'N/A':>14} {'N/A':>7} {'N/A':>6} "
              f"{'N/A':>7} {'N/A':>6} {'N/A':>6}  N/A")

=== Baseline Ingênua por empresa (persistência) ===
  Target                                   RMSE   SMAPE     R²  TheilU     DA   Cob. ColTempo
  ------------------------------ -------------- ------- ------ ------- ------ ------ --------------
  TARGET_DRE_3.01                    22,487,246   17.6%  0.959    1.00  59.7%  85.7%   TRIMESTRE
  TARGET_DRE_3.11                    21,816,677   71.2% -0.689    1.00  74.8%  85.7%   TRIMESTRE
  TARGET_EBITDA                       9,556,152   24.8%  0.946    1.00  70.1%  85.7%   TRIMESTRE
  TARGET_BPA_1                       23,200,605   17.7%  0.990    1.00  89.1%  85.7%   TRIMESTRE
  TARGET_BPA_1.01                     5,357,315   18.7%  0.971    1.00  61.3%  85.7%   TRIMESTRE
  TARGET_BPP_2.01                     6,304,194   25.5%  0.973    1.00  66.4%  85.7%   TRIMESTRE
  TARGET_BPP_2.03                     6,531,550   22.1%  0.993    1.00  74.8%  85.7%   TRIMESTRE
  TARGET_BPP_2                       23,200,605   17.7%  0.990    1.00  89.

## Etapa 3 — Algoritmos e grades de hiperparâmetros

In [9]:
# GroupKFold por empresa
gkf_ext = GroupKFold(n_splits=N_SPLITS_EXT)
gkf_int = GroupKFold(n_splits=N_SPLITS_INT)

# ── Ridge ──────────────────────────────────────────────────────────────────
est_ridge = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("ridge",   Ridge()),
])
grade_ridge = {"ridge__alpha": [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]}

# ── SVR ────────────────────────────────────────────────────────────────────
est_svr = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("svr",     SVR(kernel="rbf", max_iter=20000)),
])
grade_svr = {
    "svr__C":       [0.1, 1.0, 10.0, 100.0],
    "svr__epsilon": [0.01, 0.05, 0.1, 0.5],
    "svr__gamma":   ["scale", "auto"],
}

# ── Random Forest ──────────────────────────────────────────────────────────
est_rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf",      RandomForestRegressor(n_estimators=300,
                                       random_state=RANDOM_STATE, n_jobs=-1)),
])
grade_rf = {
    "rf__max_depth":        [None, 5, 10, 20],
    "rf__min_samples_leaf": [1, 2, 5],
    "rf__max_features":     ["sqrt", "log2"],
}

# ── Gradient Boosting ──────────────────────────────────────────────────────
est_gb = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("gb",      GradientBoostingRegressor(random_state=RANDOM_STATE)),
])
grade_gb = {
    "gb__n_estimators":  [100, 200, 300],
    "gb__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "gb__max_depth":     [3, 5],
    "gb__subsample":     [0.7, 0.8, 1.0],
}

ALGORITMOS = {
    "Ridge":            (est_ridge, grade_ridge),
    "SVR":              (est_svr,   grade_svr),
    "RandomForest":     (est_rf,    grade_rf),
    "GradientBoosting": (est_gb,    grade_gb),
}
logger.info("%d algoritmos | GroupKFold ext=%d int=%d",
            len(ALGORITMOS), N_SPLITS_EXT, N_SPLITS_INT)
print(f"✅ {len(ALGORITMOS)} algoritmos com GroupKFold(n={N_SPLITS_EXT})")

2026-05-05 10:52:24 | INFO     | 4 algoritmos | GroupKFold ext=5 int=5


✅ 4 algoritmos com GroupKFold(n=5)


## Etapa 4 — Treinamento com Nested Cross-Validation

In [10]:
def treinar_alg(nome, estimador, grade, X, y, grupos, gkf_int, gkf_ext,
                transformacao='none'):
    """
    Nested CV com GroupKFold.

    Loop interno : GridSearchCV seleciona hiperparâmetros sem vazar empresas.
    Loop externo : estima generalização no espaço original (transformação revertida).

    Transformação aplicada ao target:
        - log1p   para séries positivas e assimétricas
        - arcsinh para séries negativas/mistas
        - none    sem transformação

    Métricas calculadas: RMSE, MAE, SMAPE, R², Theil's U, Acurácia Direcional.
    """
    y_fit = target_transform(y, transformacao)

    # ── Loop interno ────────────────────────────────────────────────────
    gs = GridSearchCV(
        estimador, grade,
        cv=list(gkf_int.split(X, y_fit, grupos)),
        scoring='neg_mean_squared_error',
        refit=True, n_jobs=-1, verbose=0,
    )
    gs.fit(X, y_fit)
    melhor = gs.best_estimator_

    # ── Loop externo ────────────────────────────────────────────────────
    rmse_v, mae_v, smape_v, r2_v, theil_v, da_v = [], [], [], [], [], []

    for tr_idx, val_idx in gkf_ext.split(X, y_fit, grupos):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr        = y_fit[tr_idx]
        y_orig_val  = y[val_idx]

        melhor.fit(X_tr, y_tr)
        y_pred_raw = melhor.predict(X_val)
        y_pred     = target_inverse_transform(y_pred_raw, transformacao)

        rmse_v.append(rmse(y_orig_val, y_pred))
        mae_v.append(float(mean_absolute_error(y_orig_val, y_pred)))
        smape_v.append(smape(y_orig_val, y_pred))
        r2_v.append(float(r2_score(y_orig_val, y_pred)))
        theil_v.append(theil_u(y_orig_val, y_pred))
        da_v.append(acuracia_direcional(y_orig_val, y_pred))

    # Fit final no conjunto completo de treino
    melhor.fit(X, y_fit)

    def _m(lst): return float(np.nanmean(lst))
    def _s(lst): return float(np.nanstd(lst))

    metricas = {
        'RMSE_CV'      : _m(rmse_v),   'RMSE_CV_std'  : _s(rmse_v),
        'MAE_CV'       : _m(mae_v),
        'SMAPE_CV'     : _m(smape_v),  'SMAPE_CV_std' : _s(smape_v),
        'R2_CV'        : _m(r2_v),     'R2_CV_std'    : _s(r2_v),
        'TheilU_CV'    : _m(theil_v),
        'DA_CV'        : _m(da_v),
        'transformacao': transformacao,
        'log_transform': transformacao == 'log1p',
        'best_params'  : gs.best_params_,
    }

    flag_theil = "✅" if metricas['TheilU_CV'] < 1 else "⚠️"
    logger.info("  %-20s RMSE=%10.0f±%8.0f  SMAPE=%5.1f%%  "
                "R²=%5.3f  TheilU=%s%.3f  DA=%.1f%%  transf=%s",
                nome, metricas['RMSE_CV'], metricas['RMSE_CV_std'],
                metricas['SMAPE_CV']*100, metricas['R2_CV'],
                flag_theil, metricas['TheilU_CV'],
                metricas['DA_CV']*100, transformacao)
    print(f"  {flag_theil} {nome:<20} "
          f"RMSE={metricas['RMSE_CV']:>12,.0f}  "
          f"SMAPE={metricas['SMAPE_CV']:>5.1%}  "
          f"R²={metricas['R2_CV']:>6.3f}  "
          f"U={metricas['TheilU_CV']:.3f}  "
          f"DA={metricas['DA_CV']:.1%}")

    return melhor, metricas


# ── Execução ──────────────────────────────────────────────────────────────
resultados = {}

for target in TARGETS:
    # FIX: derivar 'transformacao' via get_target_transform — variável estava
    # sendo usada na chamada a treinar_alg sem ter sido definida no loop.
    transformacao = get_target_transform(target)
    b = baselines.get(target, {})

    print(f"\n{'='*72}")
    print(f"  TARGET: {target}  |  transform={transformacao}")
    if b:
        print(f"  Baseline → RMSE={b.get('RMSE_baseline', 0):,.0f}  "
              f"SMAPE={b.get('SMAPE_baseline', 0):.1%}  "
              f"R²={b.get('R2_baseline', 0):.3f}  "
              f"DA={b.get('DA_baseline', 0):.1%}  "
              f"Cob.={b.get('Cobertura_baseline', np.nan):.1%}")
    print(f"  {'Alg':<22} {'RMSE':>14} {'SMAPE':>7} {'R²':>7} {'TheilU':>7} {'DA':>6}")
    print(f"  {'-'*22} {'-'*14} {'-'*7} {'-'*7} {'-'*7} {'-'*6}")

    # Filtrar obs com target não-nulo
    df_t = treino[FEATURES + [target]].copy()
    df_t = df_t[df_t[target].notna()].reset_index(drop=True)
    mask = treino[target].notna()
    grupos_t = GRUPOS_TREINO[mask.values]

    X = df_t[FEATURES].values
    y = df_t[target].values

    resultados[target] = {}
    for nome, (est, grade) in ALGORITMOS.items():
        modelo, metricas = treinar_alg(
            nome, est, grade, X, y, grupos_t,
            gkf_int, gkf_ext, transformacao=transformacao,
        )
        resultados[target][nome] = (modelo, metricas)
        joblib.dump(
            {'modelo': modelo, 'transformacao': transformacao,
             'log_transform': transformacao == 'log1p', 'features': FEATURES},
            PASTA_SAIDA / f'modelo_{target}_{nome}.pkl'
        )

    logger.info("TARGET %s concluído", target)
print("\n✅ Treinamento concluído para todos os targets.")


  TARGET: TARGET_DRE_3.01  |  transform=log1p
  Baseline → RMSE=22,487,246  SMAPE=17.6%  R²=0.959  DA=59.7%  Cob.=85.7%
  Alg                              RMSE   SMAPE      R²  TheilU     DA
  ---------------------- -------------- ------- ------- ------- ------


2026-05-05 10:52:32 | INFO     |   Ridge                RMSE=  48021176±25358374  SMAPE= 84.8%  R²=-0.043  TheilU=⚠️2.714  DA=67.5%  transf=log1p


  ⚠️ Ridge                RMSE=  48,021,176  SMAPE=84.8%  R²=-0.043  U=2.714  DA=67.5%


2026-05-05 10:52:34 | INFO     |   SVR                  RMSE=  65345425±62813091  SMAPE= 75.2%  R²=-0.045  TheilU=⚠️2.798  DA=50.4%  transf=log1p


  ⚠️ SVR                  RMSE=  65,345,425  SMAPE=75.2%  R²=-0.045  U=2.798  DA=50.4%


...

  ⚠️ SVR                  RMSE=  23,347,923  SMAPE=95.4%  R²=-1.082  U=3.121  DA=50.4%


2026-05-05 11:01:54 | INFO     |   RandomForest         RMSE=  19135559±23268549  SMAPE=109.9%  R²=0.220  TheilU=⚠️2.047  DA=53.6%  transf=arcsinh


  ⚠️ RandomForest         RMSE=  19,135,559  SMAPE=109.9%  R²= 0.220  U=2.047  DA=53.6%


2026-05-05 11:02:32 | INFO     |   GradientBoosting     RMSE=  21884109±26301860  SMAPE=120.0%  R²=-0.036  TheilU=⚠️2.344  DA=54.7%  transf=arcsinh
2026-05-05 11:02:32 | INFO     | TARGET TARGET_DFC_MI_6.01 concluído


  ⚠️ GradientBoosting     RMSE=  21,884,109  SMAPE=120.0%  R²=-0.036  U=2.344  DA=54.7%

✅ Treinamento concluído para todos os targets.


## Etapa 5 — Avaliação no conjunto de teste hold-out

In [11]:
def avaliar_teste(modelo, X_te, y_te, transformacao):
    y_pred_raw = modelo.predict(X_te)
    y_pred = target_inverse_transform(y_pred_raw, transformacao)
    mask = np.isfinite(y_te) & np.isfinite(y_pred)
    yt, yp = y_te[mask], y_pred[mask]
    return {
        'RMSE_teste'  : rmse(yt, yp),
        'MAE_teste'   : float(mean_absolute_error(yt, yp)),
        'SMAPE_teste' : smape(yt, yp),
        'R2_teste'    : float(r2_score(yt, yp)),
        'TheilU_teste': theil_u(yt, yp),
        'DA_teste'    : acuracia_direcional(yt, yp),
    }


print("\n=== Avaliação no Teste Hold-out (2023–2024) ===")
metricas_teste = {}

for target in TARGETS:
    transformacao = get_target_transform(target)
    b = baselines.get(target, {})
    df_te = teste[FEATURES + [target]].copy()
    df_te = df_te[df_te[target].notna()]
    X_te  = df_te[FEATURES].values
    y_te  = df_te[target].values

    metricas_teste[target] = {}
    baseline_rmse = b.get('RMSE_baseline', np.inf)

    print(f"\n{target}  (baseline RMSE={baseline_rmse:,.0f}  "
          f"DA={b.get('DA_baseline', 0):.1%}  "
          f"Cob.={b.get('Cobertura_baseline', np.nan):.1%})")
    print(f"  {'Algoritmo':<20} {'RMSE':>14} {'SMAPE':>7} "
          f"{'R²':>7} {'TheilU':>7} {'DA':>6} {'Bateu?':>7}")
    print(f"  {'-'*20} {'-'*14} {'-'*7} {'-'*7} {'-'*7} {'-'*6} {'-'*7}")

    for nome, (modelo, _) in resultados[target].items():
        m = avaliar_teste(modelo, X_te, y_te, transformacao)
        metricas_teste[target][nome] = m
        bateu    = m['RMSE_teste'] < baseline_rmse
        theil_ok = (m['TheilU_teste'] or 1.0) < 1.0
        flag = "✅" if bateu and theil_ok else ("🟡" if bateu else "❌")
        print(f"  {flag} {nome:<18} {m['RMSE_teste']:>14,.0f} "
              f"{m['SMAPE_teste']:>7.1%} {m['R2_teste']:>7.3f} "
              f"{m['TheilU_teste']:>7.3f} {m['DA_teste']:>6.1%} "
              f"{'✅' if bateu else '❌':>7}")
        logger.info("Teste | %s | %s: RMSE=%.0f SMAPE=%.2f%% R2=%.3f TheilU=%.3f DA=%.1f%%",
                    target, nome, m['RMSE_teste'], m['SMAPE_teste']*100,
                    m['R2_teste'], m['TheilU_teste'], m['DA_teste']*100)

2026-05-05 11:02:48 | INFO     | Teste | TARGET_DRE_3.01 | Ridge: RMSE=74864081 SMAPE=78.01% R2=0.551 TheilU=1.361 DA=72.3%
2026-05-05 11:02:48 | INFO     | Teste | TARGET_DRE_3.01 | SVR: RMSE=82822868 SMAPE=62.24% R2=0.450 TheilU=1.506 DA=74.5%
2026-05-05 11:02:48 | INFO     | Teste | TARGET_DRE_3.01 | RandomForest: RMSE=57006896 SMAPE=48.27% R2=0.740 TheilU=1.036 DA=83.0%
2026-05-05 11:02:48 | INFO     | Teste | TARGET_DRE_3.01 | GradientBoosting: RMSE=44690708 SMAPE=45.02% R2=0.840 TheilU=0.812 DA=74.5%
2026-05-05 11:02:48 | INFO     | Teste | TARGET_DRE_3.11 | Ridge: RMSE=20330028 SMAPE=137.34% R2=-0.231 TheilU=1.579 DA=55.3%
2026-05-05 11:02:48 | INFO     | Teste | TARGET_DRE_3.11 | SVR: RMSE=40722190 SMAPE=109.55% R2=-3.938 TheilU=3.163 DA=59.6%



=== Avaliação no Teste Hold-out (2023–2024) ===

TARGET_DRE_3.01  (baseline RMSE=22,487,246  DA=59.7%  Cob.=85.7%)
  Algoritmo                      RMSE   SMAPE      R²  TheilU     DA  Bateu?
  -------------------- -------------- ------- ------- ------- ------ -------
  ❌ Ridge                  74,864,081   78.0%   0.551   1.361  72.3%       ❌
  ❌ SVR                    82,822,868   62.2%   0.450   1.506  74.5%       ❌
  ❌ RandomForest           57,006,896   48.3%   0.740   1.036  83.0%       ❌
  ❌ GradientBoosting       44,690,708   45.0%   0.840   0.812  74.5%       ❌

TARGET_DRE_3.11  (baseline RMSE=21,816,677  DA=74.8%  Cob.=85.7%)
  Algoritmo                      RMSE   SMAPE      R²  TheilU     DA  Bateu?
  -------------------- -------------- ------- ------- ------- ------ -------
  🟡 Ridge                  20,330,028  137.3%  -0.231   1.579  55.3%       ✅
  ❌ SVR                    40,722,190  109.5%  -3.938   3.163  59.6%       ❌


2026-05-05 11:02:49 | INFO     | Teste | TARGET_DRE_3.11 | RandomForest: RMSE=18289474 SMAPE=105.45% R2=0.004 TheilU=1.421 DA=46.8%
2026-05-05 11:02:49 | INFO     | Teste | TARGET_DRE_3.11 | GradientBoosting: RMSE=19338549 SMAPE=142.39% R2=-0.114 TheilU=1.502 DA=51.1%
2026-05-05 11:02:49 | INFO     | Teste | TARGET_EBITDA | Ridge: RMSE=30152245 SMAPE=73.95% R2=0.465 TheilU=1.399 DA=75.0%
2026-05-05 11:02:49 | INFO     | Teste | TARGET_EBITDA | SVR: RMSE=31312268 SMAPE=60.95% R2=0.423 TheilU=1.453 DA=68.2%
2026-05-05 11:02:49 | INFO     | Teste | TARGET_EBITDA | RandomForest: RMSE=13660671 SMAPE=28.87% R2=0.890 TheilU=0.633 DA=75.0%
2026-05-05 11:02:49 | INFO     | Teste | TARGET_EBITDA | GradientBoosting: RMSE=14283489 SMAPE=30.05% R2=0.880 TheilU=0.661 DA=88.6%
2026-05-05 11:02:49 | INFO     | Teste | TARGET_BPA_1 | Ridge: RMSE=120332390 SMAPE=68.37% R2=0.716 TheilU=0.988 DA=74.5%
2026-05-05 11:02:49 | INFO     | Teste | TARGET_BPA_1 | SVR: RMSE=171952793 SMAPE=67.12% R2=0.420 TheilU=

  🟡 RandomForest           18,289,474  105.5%   0.004   1.421  46.8%       ✅
  🟡 GradientBoosting       19,338,549  142.4%  -0.114   1.502  51.1%       ✅

TARGET_EBITDA  (baseline RMSE=9,556,152  DA=70.1%  Cob.=85.7%)
  Algoritmo                      RMSE   SMAPE      R²  TheilU     DA  Bateu?
  -------------------- -------------- ------- ------- ------- ------ -------
  ❌ Ridge                  30,152,245   74.0%   0.465   1.399  75.0%       ❌
  ❌ SVR                    31,312,268   61.0%   0.423   1.453  68.2%       ❌
  ❌ RandomForest           13,660,671   28.9%   0.890   0.633  75.0%       ❌
  ❌ GradientBoosting       14,283,489   30.1%   0.880   0.661  88.6%       ❌

TARGET_BPA_1  (baseline RMSE=23,200,605  DA=89.1%  Cob.=85.7%)
  Algoritmo                      RMSE   SMAPE      R²  TheilU     DA  Bateu?
  -------------------- -------------- ------- ------- ------- ------ -------
  ❌ Ridge                 120,332,390   68.4%   0.716   0.988  74.5%       ❌
  ❌ SVR                  

2026-05-05 11:02:49 | INFO     | Teste | TARGET_BPA_1.01 | RandomForest: RMSE=11525748 SMAPE=39.59% R2=0.869 TheilU=0.719 DA=70.2%
2026-05-05 11:02:49 | INFO     | Teste | TARGET_BPA_1.01 | GradientBoosting: RMSE=10595227 SMAPE=35.82% R2=0.890 TheilU=0.662 DA=68.1%
2026-05-05 11:02:49 | INFO     | Teste | TARGET_BPP_2.01 | Ridge: RMSE=23740042 SMAPE=70.92% R2=0.605 TheilU=1.208 DA=70.2%
2026-05-05 11:02:49 | INFO     | Teste | TARGET_BPP_2.01 | SVR: RMSE=29421217 SMAPE=62.54% R2=0.393 TheilU=1.498 DA=61.7%
2026-05-05 11:02:49 | INFO     | Teste | TARGET_BPP_2.01 | RandomForest: RMSE=16393063 SMAPE=41.83% R2=0.812 TheilU=0.834 DA=61.7%
2026-05-05 11:02:49 | INFO     | Teste | TARGET_BPP_2.01 | GradientBoosting: RMSE=14490545 SMAPE=42.73% R2=0.853 TheilU=0.737 DA=61.7%
2026-05-05 11:02:49 | INFO     | Teste | TARGET_BPP_2.03 | Ridge: RMSE=50902675 SMAPE=62.37% R2=0.601 TheilU=1.156 DA=63.8%
2026-05-05 11:02:49 | INFO     | Teste | TARGET_BPP_2.03 | SVR: RMSE=58298764 SMAPE=63.02% R2=0.47

  ❌ RandomForest           11,525,748   39.6%   0.869   0.719  70.2%       ❌
  ❌ GradientBoosting       10,595,227   35.8%   0.890   0.662  68.1%       ❌

TARGET_BPP_2.01  (baseline RMSE=6,304,194  DA=66.4%  Cob.=85.7%)
  Algoritmo                      RMSE   SMAPE      R²  TheilU     DA  Bateu?
  -------------------- -------------- ------- ------- ------- ------ -------
  ❌ Ridge                  23,740,042   70.9%   0.605   1.208  70.2%       ❌
  ❌ SVR                    29,421,217   62.5%   0.393   1.498  61.7%       ❌
  ❌ RandomForest           16,393,063   41.8%   0.812   0.834  61.7%       ❌
  ❌ GradientBoosting       14,490,545   42.7%   0.853   0.737  61.7%       ❌

TARGET_BPP_2.03  (baseline RMSE=6,531,550  DA=74.8%  Cob.=85.7%)
  Algoritmo                      RMSE   SMAPE      R²  TheilU     DA  Bateu?
  -------------------- -------------- ------- ------- ------- ------ -------
  ❌ Ridge                  50,902,675   62.4%   0.601   1.156  63.8%       ❌
  ❌ SVR              

2026-05-05 11:02:49 | INFO     | Teste | TARGET_BPP_2 | RandomForest: RMSE=76043996 SMAPE=29.90% R2=0.887 TheilU=0.624 DA=83.0%
2026-05-05 11:02:49 | INFO     | Teste | TARGET_BPP_2 | GradientBoosting: RMSE=42541327 SMAPE=27.76% R2=0.964 TheilU=0.349 DA=78.7%
2026-05-05 11:02:49 | INFO     | Teste | TARGET_DFC_MI_6.01 | Ridge: RMSE=22419234 SMAPE=152.45% R2=0.714 TheilU=0.958 DA=55.3%
2026-05-05 11:02:49 | INFO     | Teste | TARGET_DFC_MI_6.01 | SVR: RMSE=26128477 SMAPE=91.42% R2=0.611 TheilU=1.117 DA=61.7%
2026-05-05 11:02:49 | INFO     | Teste | TARGET_DFC_MI_6.01 | RandomForest: RMSE=23463956 SMAPE=97.93% R2=0.686 TheilU=1.003 DA=59.6%
2026-05-05 11:02:49 | INFO     | Teste | TARGET_DFC_MI_6.01 | GradientBoosting: RMSE=41340300 SMAPE=129.35% R2=0.027 TheilU=1.767 DA=70.2%


  ❌ RandomForest           76,043,996   29.9%   0.887   0.624  83.0%       ❌
  ❌ GradientBoosting       42,541,327   27.8%   0.964   0.349  78.7%       ❌

TARGET_DFC_MI_6.01  (baseline RMSE=8,086,660  DA=51.3%  Cob.=85.7%)
  Algoritmo                      RMSE   SMAPE      R²  TheilU     DA  Bateu?
  -------------------- -------------- ------- ------- ------- ------ -------
  ❌ Ridge                  22,419,234  152.4%   0.714   0.958  55.3%       ❌
  ❌ SVR                    26,128,477   91.4%   0.611   1.117  61.7%       ❌
  ❌ RandomForest           23,463,956   97.9%   0.686   1.003  59.6%       ❌
  ❌ GradientBoosting       41,340,300  129.3%   0.027   1.767  70.2%       ❌


## Etapa 6 — Feature Importance

In [12]:
def extrair_importancia(modelo, features, nome_alg):
    step = [s for s, _ in modelo.steps][-1]
    est_final = modelo.named_steps[step]
    if hasattr(est_final, 'feature_importances_'):
        imp = est_final.feature_importances_
    elif hasattr(est_final, 'coef_'):
        imp = np.abs(est_final.coef_)
    else:
        return pd.Series(dtype=float)
    return pd.Series(imp, index=features).sort_values(ascending=False)


feature_importances = {}
print("\n=== Feature Importance — Melhor Modelo por Target (critério: R²) ===")

n_t = len(TARGETS)
fig, axes = plt.subplots(n_t, 1, figsize=(11, 5*n_t))
if n_t == 1: axes = [axes]

for i, target in enumerate(TARGETS):
    melhor_nome = max(metricas_teste[target],
                      key=lambda n: metricas_teste[target][n]['R2_teste'])
    melhor_mod  = resultados[target][melhor_nome][0]
    imp = extrair_importancia(melhor_mod, FEATURES, melhor_nome)
    feature_importances[target] = {'algoritmo': melhor_nome,
                                    'importancias': imp.to_dict()}

    if not imp.empty:
        top = imp.head(min(12, len(imp)))
        colors = ['#1f4e79' if v == top.values[0] else
                  '#2e75b6' if v >= top.values[0]*0.7 else '#9dc3e6'
                  for v in top.values[::-1]]
        axes[i].barh(range(len(top)), top.values[::-1], color=colors, alpha=0.9)
        axes[i].set_yticks(range(len(top)))
        axes[i].set_yticklabels(top.index[::-1], fontsize=9)
        axes[i].set_title(f"{target.replace('TARGET_', '')} — {melhor_nome} "
                           f"(R²={metricas_teste[target][melhor_nome]['R2_teste']:.3f})",
                           fontsize=11, fontweight='bold')
        axes[i].set_xlabel('Importância Relativa')
        axes[i].grid(axis='x', alpha=0.3)
        for j, v in enumerate(top.values[::-1]):
            axes[i].text(v + imp.max()*0.005, j, f'{v:.3f}', va='center', fontsize=8)
    print(f"  {target}: {melhor_nome} | top3={list(imp.head(3).index)}")

plt.suptitle('Feature Importance — Melhor Modelo por Target',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PASTA_SAIDA / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Salvo: feature_importance.png")


=== Feature Importance — Melhor Modelo por Target (critério: R²) ===
  TARGET_DRE_3.01: GradientBoosting | top3=['div_liquida', 'EBITDA', 'setor_Petróleo']
  TARGET_DRE_3.11: RandomForest | top3=['macro_pib_tri', 'margem_liquida_yoy', 'div_liquida']
  TARGET_EBITDA: RandomForest | top3=['EBITDA', 'div_liquida', 'FCF']
  TARGET_BPA_1: GradientBoosting | top3=['div_liquida', 'EBITDA', 'macro_pib_tri']
  TARGET_BPA_1.01: GradientBoosting | top3=['div_liquida', 'EBITDA', 'fco_receita']
  TARGET_BPP_2.01: GradientBoosting | top3=['div_liquida', 'EBITDA', 'liquidez_corrente']
  TARGET_BPP_2.03: GradientBoosting | top3=['EBITDA', 'div_liquida', 'giro_ativo']
  TARGET_BPP_2: GradientBoosting | top3=['div_liquida', 'EBITDA', 'macro_pib_tri']
  TARGET_DFC_MI_6.01: Ridge | top3=['conversao_caixa', 'fco_receita', 'setor_Varejo']
  ✅ Salvo: feature_importance.png


## Etapa 7 — Curvas de Aprendizado

In [13]:
print("\nGerando curvas de aprendizado...")

n_t = len(TARGETS)
fig, axes = plt.subplots(1, n_t, figsize=(7*n_t, 5))
if n_t == 1: axes = [axes]

for i, target in enumerate(TARGETS):
    transformacao = get_target_transform(target)
    df_t = treino[FEATURES + [target]].copy()
    df_t = df_t[df_t[target].notna()].reset_index(drop=True)
    mask = treino[target].notna()
    grupos_t = GRUPOS_TREINO[mask.values]
    X = df_t[FEATURES].values
    y = target_transform(df_t[target].values, transformacao)

    # Curvas de aprendizado usam o melhor modelo pelo mesmo critério da Etapa 6 (R²)
    melhor_nome = max(metricas_teste[target],
                      key=lambda n: metricas_teste[target][n]['R2_teste'])
    melhor_mod  = resultados[target][melhor_nome][0]

    try:
        sizes, tr_sc, val_sc = learning_curve(
            melhor_mod, X, y,
            cv=list(gkf_ext.split(X, y, grupos_t)),
            scoring='r2',
            train_sizes=np.linspace(0.2, 1.0, 6),
            n_jobs=-1,
        )
        ax = axes[i]
        ax.plot(sizes, tr_sc.mean(1), 'o-', label='Treino',     color='#1f4e79', lw=2)
        ax.fill_between(sizes, tr_sc.mean(1)-tr_sc.std(1),
                         tr_sc.mean(1)+tr_sc.std(1), alpha=0.12, color='#1f4e79')
        ax.plot(sizes, val_sc.mean(1), 's--', label='Validação', color='#c0392b', lw=2)
        ax.fill_between(sizes, val_sc.mean(1)-val_sc.std(1),
                         val_sc.mean(1)+val_sc.std(1), alpha=0.12, color='#c0392b')
        gap  = tr_sc.mean(1)[-1] - val_sc.mean(1)[-1]
        diag = ('overfitting'  if gap > 0.15 else
                'underfitting' if val_sc.mean(1)[-1] < 0.3 else 'OK')
        ax.set_title(f"{target.replace('TARGET_', '')}\n{melhor_nome}",
                      fontsize=10, fontweight='bold')
        ax.set_xlabel(f'Tamanho do treino  |  Gap={gap:.2f} → {diag}', fontsize=9)
        ax.set_ylabel('R²')
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)
        logger.info("Curva %s/%s: gap=%.3f diag=%s", target, melhor_nome, gap, diag)
    except Exception as e:
        logger.warning("Curva de aprendizado falhou %s/%s: %s", target, melhor_nome, e)
        axes[i].text(0.5, 0.5, 'Erro na curva\n'+str(e)[:60],
                     ha='center', va='center', transform=axes[i].transAxes, fontsize=9)

plt.suptitle('Curvas de Aprendizado — Diagnóstico de Bias/Variância',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PASTA_SAIDA / 'curvas_aprendizado.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Salvo: curvas_aprendizado.png")


Gerando curvas de aprendizado...


2026-05-05 11:03:53 | INFO     | Curva TARGET_DRE_3.01/GradientBoosting: gap=0.521 diag=overfitting
2026-05-05 11:03:57 | INFO     | Curva TARGET_DRE_3.11/RandomForest: gap=1.268 diag=overfitting
2026-05-05 11:04:01 | INFO     | Curva TARGET_EBITDA/RandomForest: gap=0.164 diag=overfitting
2026-05-05 11:04:03 | INFO     | Curva TARGET_BPA_1/GradientBoosting: gap=0.215 diag=overfitting
2026-05-05 11:04:05 | INFO     | Curva TARGET_BPA_1.01/GradientBoosting: gap=0.312 diag=overfitting
2026-05-05 11:04:07 | INFO     | Curva TARGET_BPP_2.01/GradientBoosting: gap=0.356 diag=overfitting
2026-05-05 11:04:10 | INFO     | Curva TARGET_BPP_2.03/GradientBoosting: gap=0.445 diag=overfitting
2026-05-05 11:04:13 | INFO     | Curva TARGET_BPP_2/GradientBoosting: gap=0.215 diag=overfitting
2026-05-05 11:04:13 | INFO     | Curva TARGET_DFC_MI_6.01/Ridge: gap=0.132 diag=underfitting


  ✅ Salvo: curvas_aprendizado.png


## Etapa 8 — Análise de Resíduos

In [14]:
print("\nGerando análise de resíduos...")

n_t = len(TARGETS)
fig, axes = plt.subplots(n_t, 2, figsize=(14, 5*n_t))
if n_t == 1: axes = axes.reshape(1, -1)

for i, target in enumerate(TARGETS):
    transformacao = get_target_transform(target)
    df_te = teste[FEATURES + [target]].copy()
    df_te = df_te[df_te[target].notna()]
    X_te  = df_te[FEATURES].values
    y_te  = df_te[target].values

    # Análise de resíduos usa o mesmo melhor modelo das outras etapas (R²)
    melhor_nome = max(metricas_teste[target],
                      key=lambda n: metricas_teste[target][n]['R2_teste'])
    mod    = resultados[target][melhor_nome][0]
    y_pred = target_inverse_transform(mod.predict(X_te), transformacao)
    residuos = y_te - y_pred

    # Predito × Observado
    ax1 = axes[i, 0]
    lim = max(np.nanmax(np.abs(y_te)), np.nanmax(np.abs(y_pred))) * 1.05
    ax1.scatter(y_pred, y_te, alpha=0.45, s=18, color='#1f4e79', edgecolors='none')
    ax1.plot([0, lim], [0, lim], 'r--', lw=1.5)
    ax1.set_xlabel('Predito (R$ mil)')
    ax1.set_ylabel('Observado (R$ mil)')
    ax1.set_title(f"{target.replace('TARGET_', '')} — {melhor_nome}\nPredito × Observado",
                   fontsize=10, fontweight='bold')
    r2_val = metricas_teste[target][melhor_nome]['R2_teste']
    ax1.text(0.05, 0.92, f'R²={r2_val:.3f}', transform=ax1.transAxes, fontsize=9,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    # Resíduos × Predito
    ax2 = axes[i, 1]
    ax2.scatter(y_pred, residuos, alpha=0.45, s=18, color='#744210', edgecolors='none')
    ax2.axhline(0, color='r', lw=1.5, ls='--')
    ax2.axhline( np.std(residuos), color='gray', lw=1, ls=':', alpha=0.7)
    ax2.axhline(-np.std(residuos), color='gray', lw=1, ls=':', alpha=0.7)
    ax2.set_xlabel('Predito (R$ mil)')
    ax2.set_ylabel('Resíduo (R$ mil)')
    ax2.set_title(f'Resíduos  |  skew={pd.Series(residuos).skew():.2f}  '
                   f'σ={np.std(residuos):,.0f}', fontsize=10, fontweight='bold')

plt.suptitle('Análise de Resíduos — Conjunto de Teste 2023–2024',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PASTA_SAIDA / 'analise_residuos.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Salvo: analise_residuos.png")


Gerando análise de resíduos...
  ✅ Salvo: analise_residuos.png


## Etapa 9 — Persistência completa

In [15]:
rows_cv, rows_te = [], []

for target, algs in resultados.items():
    b = baselines.get(target, {})
    for alg, (_, m) in algs.items():
        rows_cv.append({
            'Target'       : target,
            'Algoritmo'    : alg,
            'RMSE_CV'      : m['RMSE_CV'],
            'RMSE_CV_std'  : m.get('RMSE_CV_std'),
            'SMAPE_CV'     : m['SMAPE_CV'],
            'R2_CV'        : m['R2_CV'],
            'TheilU_CV'    : m.get('TheilU_CV'),
            'DA_CV'        : m.get('DA_CV'),
            'transformacao': m.get('transformacao'),
            'log_transform': m['log_transform'],
            'best_params'  : str(m['best_params']),
        })
        mt = metricas_teste[target][alg]
        rows_te.append({
            'Target'         : target,
            'Algoritmo'      : alg,
            'RMSE_teste'     : mt['RMSE_teste'],
            'MAE_teste'      : mt['MAE_teste'],
            'SMAPE_teste'    : mt['SMAPE_teste'],
            'R2_teste'       : mt['R2_teste'],
            'TheilU_teste'   : mt.get('TheilU_teste'),
            'DA_teste'       : mt.get('DA_teste'),
            'RMSE_baseline'  : b.get('RMSE_baseline'),
            'Bateu_baseline' : mt['RMSE_teste'] < b.get('RMSE_baseline', np.inf),
            'TheilU_ok'      : (mt.get('TheilU_teste', 1.0) or 1.0) < 1.0,
        })

df_cv = pd.DataFrame(rows_cv)
df_te = pd.DataFrame(rows_te)

# FIX: critério de seleção do melhor modelo consistente com Etapas 6/7/8 (R²)
melhores = {
    t: df_te[df_te['Target'] == t]
         .sort_values('R2_teste', ascending=False)
         .iloc[0]['Algoritmo']
    for t in TARGETS
}

df_cv.to_csv(PASTA_SAIDA / 'resultados_cv.csv',    index=False)
df_te.to_csv(PASTA_SAIDA / 'resultados_teste.csv', index=False)

with open(PASTA_SAIDA / 'resultados_cv.pkl',       'wb') as f: pickle.dump(resultados, f)
with open(PASTA_SAIDA / 'metricas_teste.pkl',      'wb') as f: pickle.dump(metricas_teste, f)
with open(PASTA_SAIDA / 'baselines.pkl',           'wb') as f: pickle.dump(baselines, f)
with open(PASTA_SAIDA / 'feature_importances.pkl', 'wb') as f: pickle.dump(feature_importances, f)
with open(PASTA_SAIDA / 'melhores_modelos.pkl',    'wb') as f: pickle.dump(melhores, f)

# FIX: versão corrigida para 'V5_9targets_split_temporal'
relatorio = {
    'versao'            : 'V5_9targets_split_temporal',
    'ano_corte'         : ANO_CORTE,
    'n_treino'          : int(len(treino)),
    'n_teste'           : int(len(teste)),
    'algoritmos'        : list(ALGORITMOS.keys()),
    'targets'           : TARGETS,
    'log_targets'       : list(LOG_TARGETS),
    'arcsinh_targets'   : list(ARCSINH_TARGETS),
    'target_transforms' : {t: get_target_transform(t) for t in TARGETS},
    'features'          : FEATURES,
    'melhores'          : melhores,
    'baselines'         : {
        t: {k: float(v) for k, v in b.items()
            if isinstance(v, (int, float, np.floating))}
        for t, b in baselines.items()
    },
}
with open(PASTA_SAIDA / 'relatorio_modelagem.json', 'w', encoding='utf-8') as f:
    json.dump(relatorio, f, indent=2, ensure_ascii=False, default=str)

print("\n" + "═"*72)
print("  RESUMO FINAL — Script 3 V5 (baseline por empresa)")
print("═"*72)
print(f"  Treino  : {len(treino):,} obs (≤{ANO_CORTE}) | "
      f"DFP={(treino['ORIGEM']=='DFP').sum()} | ITR={(treino['ORIGEM']=='ITR').sum()}")
print(f"  Teste   : {len(teste):,} obs (≥{ANO_CORTE+1}) | "
      f"DFP={(teste['ORIGEM']=='DFP').sum()}  | ITR={(teste['ORIGEM']=='ITR').sum()}")
print(f"  Modelos : {len(TARGETS) * len(ALGORITMOS)} ({len(TARGETS)} targets × {len(ALGORITMOS)} algoritmos)")
print(f"  Targets : {len(TARGETS)} (3 primários + 5 balanço + 1 caixa)")
print(f"  Métricas: RMSE, MAE, SMAPE, R², Theil's U, Acurácia Direcional")
print(f"  Melhores por R²:")
for t, alg in melhores.items():
    m = metricas_teste[t][alg]
    b = baselines.get(t, {})
    bateu = m['RMSE_teste'] < b.get('RMSE_baseline', np.inf)
    print(f"    {t:<35} {alg:<20} R²={m['R2_teste']:.3f}  "
          f"RMSE={m['RMSE_teste']:,.0f}  "
          f"SMAPE={m['SMAPE_teste']:.1%}  U={m['TheilU_teste']:.3f}  "
          f"{'✅' if bateu else '❌'}")
print("═"*72)
print("  ✅ Pronto para Script 4 (Avaliação + Z'')")
print("═"*72)



════════════════════════════════════════════════════════════════════════
  RESUMO FINAL — Script 3 V5 (baseline por empresa)
════════════════════════════════════════════════════════════════════════
  Treino  : 713 obs (≤2022) | DFP=181 | ITR=532
  Teste   : 168 obs (≥2023) | DFP=24  | ITR=144
  Modelos : 36 (9 targets × 4 algoritmos)
  Targets : 9 (3 primários + 5 balanço + 1 caixa)
  Métricas: RMSE, MAE, SMAPE, R², Theil's U, Acurácia Direcional
  Melhores por R²:
    TARGET_DRE_3.01                     GradientBoosting     R²=0.840  RMSE=44,690,708  SMAPE=45.0%  U=0.812  ❌
    TARGET_DRE_3.11                     RandomForest         R²=0.004  RMSE=18,289,474  SMAPE=105.5%  U=1.421  ✅
    TARGET_EBITDA                       RandomForest         R²=0.890  RMSE=13,660,671  SMAPE=28.9%  U=0.633  ❌
    TARGET_BPA_1                        GradientBoosting     R²=0.964  RMSE=42,541,327  SMAPE=27.8%  U=0.349  ❌
    TARGET_BPA_1.01                     GradientBoosting     R²=0.890  RMSE=10,5